In [ ]:
!pip install kagglehub
!pip install tiktoken
!pip install tqdm
!pip install datasets
!git clone https://github.com/tylerksoong/transformers.git
!cp -r transformers/transformer/ ./

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import pandas as pd
import copy
from tqdm import tqdm
import tiktoken
import kagglehub
import gc

PAD_ID = 50257
DEVICE = "cuda"

In [ ]:
from datasets import load_dataset

train_raw = load_dataset("roneneldan/TinyStories", split="train")
valid_raw = load_dataset("roneneldan/TinyStories", split="validation")

In [ ]:
print(train_raw)

In [ ]:
class TinyStories(Dataset):
    def __init__(self, df):
        self.df = df
        self.tokenizer = tiktoken.get_encoding("r50k_base")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df[idx]["text"]
        tokens = self.tokenizer.encode(text)
        input_ids = torch.tensor(tokens[:-1])
        target = torch.tensor(tokens[1:])
        return input_ids, target

In [ ]:
train_dataset=TinyStories(train_raw)
valid_dataset=TinyStories(valid_raw)

In [ ]:
def collate_fn(batch):

    input_ids, target = zip(*batch)
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=PAD_ID)
    target = pad_sequence(target, batch_first=True, padding_value=PAD_ID)

    return input_ids, target


In [ ]:
print(train_dataset.__getitem__(0))

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_dataloader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn
)

In [ ]:
scaler = GradScaler(DEVICE)
import time


def train_step(model, optimizer, dataloader, epoch, device=DEVICE):
    t = time.perf_counter()
    model.train()
    train_loss = 0

    avg_loss = 0.0

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}", dynamic_ncols=True)
    for idx, (inputs, target) in enumerate(dataloader, start = 1):
        with autocast(device):
            inputs = inputs.to(device)
            target = target.to(device)
            logits = model(inputs)
            loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=PAD_ID, label_smoothing=0.1)

            batch_loss = loss.item()
            train_loss += batch_loss
            avg_loss = train_loss/idx

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        #total_weight_values = sum(p.abs().sum() for p in model.parameters())
        #print(total_weight_values)
        scaler.update()
        optimizer.zero_grad()
        print(f"Step {idx} | Train Loss:", avg_loss)


        if idx % 100 == 0:
            checkpoint = {
                'model_state_dict' : model.state_dict(),
                'optimizer_state_dict' : optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }
            torch.save(obj=checkpoint, f=f"weights/BlockFormer_{idx}_steps.pth")
        del inputs, target, loss, logits

    print(f"Epoch {epoch} Step {idx} | Train Loss: {avg_loss:.4f}")
    torch.accelerator.empty_cache()
    return avg_loss



def test_step(model, epoch, dataloader, device=DEVICE):
    t = time.perf_counter()
    model.eval()
    test_loss, total_tokens = 0, 0

    avg_loss = 0.0

    with torch.no_grad():
        for idx, (inputs, target) in enumerate(dataloader, start = 1):
            with autocast(device):
                inputs = inputs.to(device)
                target = target.to(device)
                logits = model(inputs)
                loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=PAD_ID, label_smoothing=0.1, reduction='sum')

                batch_loss = loss.item()
                test_loss += batch_loss
                batch_tokens = (target != PAD_ID).sum().item()
                total_tokens += batch_tokens

            # batch_ppl = torch.exp(torch.tensor(batch_loss / batch_tokens))
            # progress_bar.set_postfix(batch_ppl=f"{batch_ppl:.4f}")

            del inputs, target, loss, logits


    avg_loss = test_loss / total_tokens
    avg_nll = test_loss / total_tokens
    ppl = torch.exp(torch.tensor(avg_nll))

    print(f"Epoch {epoch} | Test Loss: {avg_loss:.4f} | Test Preplexity: {ppl:.4f}")
    torch.accelerator.empty_cache()
    return avg_loss, ppl

In [ ]:
from transformer.transformer import Transformer

ChatGPT6 = Transformer(DEVICE)
ChatGPT6 = ChatGPT6.to(DEVICE)



In [ ]:
current = 0
num_epochs = 1

start = 1 + current
epochs = num_epochs + current + 1

print(f"Running from {start} to {epochs-1}")

model = nn.DataParallel(ChatGPT6)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=3e-5
)

torch.manual_seed(42)
torch.cuda.manual_seed(42)

print("Ready to train!!")

for epoch in range(start, epochs):
    print("####### Epoch:", epoch)
    print("NOW: Training")

    train_step(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        dataloader=train_dataloader,
        device=DEVICE
    )

    print("NOW: Testing")
    test_step(
        model=model,
        epoch=epoch,
        dataloader=test_dataloader,
    )
    # scheduler.step()
    checkpoint = {
        'model_state_dict' : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict()
    }
    torch.save(obj=checkpoint, f=f"weights/BlockFormer_{epoch}_epochs.pth")